# Implementation of basic NEAT algorithm.

In [1]:
import numpy as np
import networkx as nx
from __future__ import annotations
import copy

In [98]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class GenomeConfig:
    use_small_distance_heuristic: bool = True
    cross_p_inherit_weaker: float = 0.5
    cross_p_stay_disabled: float = 0.75
    mut_p_add_node: float = 0.05
    mut_p_add_link: float = 0.1
    mut_p_replace: float = 0.1 # 1 - p_replace = p_pertubate
    mut_pertubation_range: float = 2.0
    mut_max_weight: float = 8.0
    mut_min_weight: float = -8.0
    mut_clip_weights: bool = True
    #activation_fun: Callable = lambda W, x: 1 / (1 + np.exp(np.clip(-4.9 * (W @ x), -500, 500))) # NEAT steepened sigmoid
    activation_fun: Callable = lambda W, x: np.tanh(W @ x)

In [24]:
from enum import Enum

class NodeType(Enum):
    INPUT = 0
    OUTPUT = 1
    HIDDEN = 2

class Individual:
    """
    Represents one solution/network/genome/individual.
    
    nodes is a dict with {innovation_number : NodeType}
    connections is a dict with {innovation_number : Connection}
    """
    def __init__(self):
        self.config = GenomeConfig()
        self.network = nx.DiGraph()
        self.fitness = 0
        self.W = None

    def add_node_gene(self, node_id: int, node_type: NodeType):
        self.network.add_node(node_id, type=node_type)

    def add_connection_gene(self, in_node: int, out_node: int, innov: int, weight: float, enabled: bool):
        self.network.add_edge(in_node, out_node, innov=innov, weight=weight, enabled=enabled)

    def size(self) -> int:
        return self.network.number_of_nodes() + self.network.number_of_edges()

    def clone(self) -> Individual:
        child = Individual()
        child.config = self.config
        child.network = self.network.copy()
        child.compile_network()
        return child

    def _get_innov_dict(self):
        """Helper to get O(1) innovation lookups for crossover/distance."""
        return {attr['innov']: (u, v, attr) 
                for u, v, attr in self.network.edges(data=True)}

    def distance(self, other: Individual, c1=1.0, c2=0.4) -> float:
        """
        Compute *Compatibiliy Distance*, we currently ignore node genes and the distiction between excess and disjoint genes.
        """
        genes1 = self._get_innov_dict()
        genes2 = other._get_innov_dict()
        innovation_numbers1 = set(genes1.keys())
        innovation_numbers2 = set(genes2.keys())
        
        matching_keys = innovation_numbers1 & innovation_numbers2
        non_homologous_count = len(innovation_numbers1 ^ innovation_numbers2)

        if matching_keys:
            w1 = np.array([genes1[k][2]['weight'] for k in matching_keys])
            w2 = np.array([genes2[k][2]['weight'] for k in matching_keys])
            weight_diff = np.mean(np.abs(w1 - w2))
        else:
            weight_diff = 0.0
    
        N = max(len(innovation_numbers1), len(innovation_numbers2))
        if self.config.use_small_distance_heuristic:
            N = 1 if N < 20 else N 

        return (c1 * non_homologous_count / N) + (c2 * weight_diff)

    def crossover(self, other: Individual) -> Individual:
        """
        Innovation number aligned crossover with tunable inheritance probabilities.
        """
        child = Individual()
        fitter, weaker = (self, other) if self.fitness >= other.fitness else (other, self)

        genes_fitter = fitter._get_innov_dict()
        genes_weaker = weaker._get_innov_dict()

        child.network.add_nodes_from(fitter.network.nodes(data=True))

        for innov_num, (u, v, attr) in genes_fitter.items():
            child_attr = copy.deepcopy(attr)

            if innov_num in genes_weaker:
                weaker_attr = genes_weaker[innov_num][2]
                
                if np.random.random() < self.config.cross_p_inherit_weaker:
                    child_attr['weight'] = weaker_attr['weight']
                    
                if not attr['enabled'] or not weaker_attr['enabled']:
                    child_attr['enabled'] = np.random.random() > self.config.cross_p_stay_disabled 
                    
            child.network.add_edge(u, v, **child_attr)
        
        return child

    def mutate_weights(self):
        edges = list(self.network.edges(data=True))
        n_conns = len(edges)
        
        perturbations = np.random.uniform(-self.config.mut_pertubation_range, self.config.mut_pertubation_range, n_conns)
        replacements = np.random.uniform(self.config.mut_min_weight, self.config.mut_max_weight, n_conns)
        action_probs = np.random.rand(n_conns)
    
        for i, (u, v, attr) in enumerate(edges):
            if action_probs[i] < self.config.mut_p_replace:
                attr['weight'] = replacements[i]
            else:
                attr['weight'] += perturbations[i]
                if self.config.mut_clip_weights:
                    attr['weight'] = max(self.config.mut_min_weight, min(self.config.mut_max_weight, attr['weight']))

    def mutate_genes(self, get_link_innovation: Callable, get_node_innovation: Callable):
        if np.random.random() < self.config.mut_p_add_link:
            nodes = list(self.network.nodes(data=True))
            
            in_node, in_attr = nodes[np.random.choice(len(nodes))]
            out_node, out_attr = nodes[np.random.choice(len(nodes))]
            
            is_valid_type = in_attr['type'] != NodeType.OUTPUT and out_attr['type'] != NodeType.INPUT
            is_not_loop = in_node != out_node
            is_new = not self.network.has_edge(in_node, out_node)
            is_not_looping = not nx.has_path(self.network, out_node, in_node)
            
            if is_valid_type and is_not_loop and is_new and is_not_looping:
                self.add_connection_gene(
                    in_node, out_node,
                    innov=get_link_innovation(in_node, out_node),
                    weight=np.random.uniform(self.config.mut_min_weight, self.config.mut_max_weight),
                    enabled=True
                )
        
        if np.random.random() < self.config.mut_p_add_node:
            # split: N_1 -> N_2 to N_1 -> N_new -> N_2
            enabled_edges = [(u, v, attr) for u, v, attr in self.network.edges(data=True) if attr['enabled']]
            if not enabled_edges:
                return
            
            n_1, n_2, attr = enabled_edges[np.random.choice(len(enabled_edges))]
            
            attr['enabled'] = False
            old_weight = attr['weight']
            link_innov = attr['innov']
            
            new_node_id = get_node_innovation(link_innov)
            self.add_node_gene(new_node_id, NodeType.HIDDEN)
            self.add_connection_gene(
                in_node=n_1, out_node=new_node_id,
                innov=get_link_innovation(n_1, new_node_id),
                weight=1.0, enabled=True
            )
            self.add_connection_gene(
                in_node=new_node_id, out_node=n_2,
                innov=get_link_innovation(new_node_id, n_2),
                weight=old_weight, enabled=True
            )

    def compile_network(self):
        """
        Compile network into adjacency matrix of weights between nodes.
        """
        self._node_idx_map = {key: i for i, key in enumerate(self.network.nodes())}
        n_nodes = len(self._node_idx_map)
        self.W = np.zeros((n_nodes, n_nodes))

        # used in forward_pass input/output
        self._input_indices = [
            self._node_idx_map[n] for n, attr in self.network.nodes(data=True) 
            if attr['type'] == NodeType.INPUT
        ]
        self._output_indices = [
            self._node_idx_map[n] for n, attr in self.network.nodes(data=True) 
            if attr['type'] == NodeType.OUTPUT
        ]
        
        for u, v, attr in self.network.edges(data=True):
            if attr['enabled']:
                u_idx, v_idx = self._node_idx_map[u], self._node_idx_map[v]
                self.W[v_idx][u_idx] = attr['weight']

        self._network_depth = nx.dag_longest_path_length(self.network, weight=None) + 1
    
    def forward_pass(self, inputs: np.ndarray) -> np.ndarray:
        state = np.zeros(len(self._node_idx_map))

        if len(inputs) != len(self._input_indices):
            raise ValueError("Number of input nodes doesn't match")
        state[:len(inputs)] = inputs 
        
        for _ in range(self._network_depth):
            new_state = self.config.activation_fun(self.W, state)
            new_state[:len(inputs)] = inputs
            state = new_state
        
        return state[self._output_indices]

In [49]:
@dataclass
class PopulationConfig:
    dropoff_age: int = 10
    p_selection: float = 0.2
    p_mutation_without_cross: float = 0.25
    p_interspecies_mate: float = 0.001
    n_min_elite_survival: int = 3
    use_random_representatives: bool = False

In [50]:
from concurrent.futures import ProcessPoolExecutor
from collections import Counter, defaultdict

class Population:
    """
    This class holds a population of Individuals.
    """
    def __init__(self, size:int, compatibility_threshold: float, cluster_species: Callable):
        self.config = PopulationConfig()
        self.size = size
        self.individuals = []
        self.cluster_species = cluster_species
        self.compatibility_threshold = compatibility_threshold
        # track global innovation numbers for shared genes
        self.global_innov_counter = 0
        self.global_node_counter = 0
        self.current_generation_innovations = {}
        # tracking drop-off age
        self.species_history = defaultdict(lambda: {'max_fitness': -float('inf'), 'stagnant_gens': 0})
        # track representatives for clustering in next generation
        self.previous_representatives = {}

    def _get_link_innovation(self, in_node: int, out_node: int) -> int:
        key = ('link', in_node, out_node)
        if key not in self.current_generation_innovations:
            self.global_innov_counter += 1
            self.current_generation_innovations[key] = self.global_innov_counter
        return self.current_generation_innovations[key]

    def _get_node_innovation(self, link_to_split: int) -> int:
        key = ('node', link_to_split)
        if key not in self.current_generation_innovations:
            self.global_node_counter += 1
            self.current_generation_innovations[key] = self.global_node_counter
        return self.current_generation_innovations[key]

    def reset_generation_innovations(self):
        self.current_generation_innovations.clear()

    def select_reproduce_mutate(self):
        # 1. Cluster and Allocations
        clusters = self.cluster_species(self.individuals, self.previous_representatives, self.compatibility_threshold)
        n_per_cluster = Counter(clusters)
        
        fitnesses = np.array([max(0.0001, ind.fitness) for ind in self.individuals]) # Prevent zero/negative math errors
        shared_fitnesses = fitnesses / np.array([n_per_cluster[c] for c in clusters])

        # Sum fitness per cluster and enforce dropoff age
        fitness_per_cluster = {}
        for cluster in set(clusters):
            cluster_inds = [ind for ind, c in zip(self.individuals, clusters) if c == cluster]
            max_fit = max(ind.fitness for ind in cluster_inds)
            
            history = self.species_history[cluster]
            if max_fit > history['max_fitness']:
                history['max_fitness'] = max_fit
                history['stagnant_gens'] = 0
            else:
                history['stagnant_gens'] += 1
                
            is_stagnant = history['stagnant_gens'] >= self.config.dropoff_age
            is_best_species = max(ind.fitness for ind in self.individuals) == max_fit
            
            if is_stagnant and not is_best_species:
                fitness_per_cluster[cluster] = 0.0001
            else:
                fitness_per_cluster[cluster] = sum(shared_fitnesses[np.array(clusters) == cluster])

        # Offspring Allocation
        total_adjusted_fitness = sum(fitness_per_cluster.values())
        allocation_per_cluster = {
            c: int((fit / total_adjusted_fitness) * self.size) 
            for c, fit in fitness_per_cluster.items()
        }
        
        # Handle rounding errors
        missing_allocations = self.size - sum(allocation_per_cluster.values())
        if missing_allocations > 0:
            best_cluster = max(fitness_per_cluster, key=fitness_per_cluster.get)
            allocation_per_cluster[best_cluster] += missing_allocations

        # 2. Reproduction and Mutation
        next_generation_individuals = []
        all_survivors = []
        cluster_survivor_map = {}
        
        for cluster in set(clusters):
            cluster_inds = [ind for ind, c in zip(self.individuals, clusters) if c == cluster]
            cluster_inds.sort(key=lambda ind: ind.fitness, reverse=True)
            n_survivors = max(1, int(len(cluster_inds) * self.config.p_selection))
            
            survivors = cluster_inds[:n_survivors]
            cluster_survivor_map[cluster] = survivors
            all_survivors.extend(survivors)

        for cluster, allocation in allocation_per_cluster.items():
            if allocation == 0:
                continue
                
            survivors = cluster_survivor_map[cluster]
            offspring_count = 0
            
            # Elite survival
            if len(survivors) >= self.config.n_min_elite_survival:
                next_generation_individuals.append(survivors[0].clone())
                offspring_count += 1

            while offspring_count < allocation:
                if np.random.random() < self.config.p_mutation_without_cross:
                    # Clone and Mutate
                    parent = np.random.choice(survivors)
                    child = parent.clone()
                else:
                    # Crossover
                    parent1, parent2 = np.random.choice(survivors, 2)
                    
                    # INTERSPECIES MATING
                    if np.random.random() < self.config.p_interspecies_mate:
                        parent2 = np.random.choice(all_survivors)
                        
                    child = parent1.crossover(parent2)

                child.mutate_weights()
                child.mutate_genes(self._get_link_innovation, self._get_node_innovation)
                child.compile_network()
                
                next_generation_individuals.append(child)
                offspring_count += 1

        # Update representatives for next generation clustering
        self.previous_representatives.clear()
        for cluster in set(clusters):
            survivors = cluster_survivor_map.get(cluster, [])
            if not survivors:
                continue # Species went extinct
                
            if self.config.use_random_representatives:
                self.previous_representatives[cluster] = np.random.choice(survivors)
            else:
                self.previous_representatives[cluster] = survivors[0]

        self.individuals = next_generation_individuals
        self.reset_generation_innovations()


In [75]:
# split evalute_fitness for readability
class Population(Population):
    def evaluate_fitness(self, vector_env, action_mapper=None, max_steps=1000, n_runs=3):
        """
        Evaluates the population concurrently using a Gymnasium VectorEnv.
        Runs each agent 'n_runs' times and averages their fitness.
        """
        if vector_env.num_envs != self.size:
            raise ValueError(f"VectorEnv must have exactly {self.size} environments to match population size.")
        
        total_fitness_scores = np.zeros(self.size, dtype=float)

        for run in range(n_runs):
            obs, info = vector_env.reset()
            
            finished = np.zeros(self.size, dtype=bool)
            current_run_scores = np.zeros(self.size, dtype=float)

            for _ in range(max_steps):
                if finished.all():
                    break

                actions = []
                for i, ind in enumerate(self.individuals):
                    if not finished[i]:
                        raw_action = ind.forward_pass(obs[i])
                        action = action_mapper(raw_action) if action_mapper else raw_action
                        actions.append(action)
                    else:
                        # dummy action
                        actions.append(vector_env.single_action_space.sample())
                
                obs, rewards, terminated, truncated, infos = vector_env.step(np.array(actions))

                dones = terminated | truncated
                current_run_scores += rewards * (~finished)
                finished = finished | dones

            total_fitness_scores += current_run_scores

        avg_fitness_scores = total_fitness_scores / n_runs

        for ind, fit in zip(self.individuals, avg_fitness_scores):
            ind.fitness = fit

In [7]:
def create_population(size: int, n_inputs: int, n_outputs: int, compatibility_threshold:float, cluster_func: callable) -> Population:
    """
    Initializes a NEAT population with minimal, fully-connected networks.
    Inputs are connected to all outputs with random weights.
    """
    pop = Population(size=size, compatibility_threshold=compatibility_threshold, cluster_species=cluster_func)
    pop.global_node_counter = n_inputs + n_outputs - 1
    
    for _ in range(size):
        ind = Individual()
        
        # I/O Nodes
        for i in range(n_inputs):
            ind.add_node_gene(node_id=i, node_type=NodeType.INPUT)
        for j in range(n_outputs):
            out_node_id = n_inputs + j
            ind.add_node_gene(node_id=out_node_id, node_type=NodeType.OUTPUT)
            
        # Connections
        for in_node in range(n_inputs):
            for j in range(n_outputs):
                out_node = n_inputs + j
                
                innov_num = pop._get_link_innovation(in_node, out_node)
                
                weight = np.random.uniform(
                    ind.config.mut_min_weight, 
                    ind.config.mut_max_weight
                )
                
                ind.add_connection_gene(
                    in_node=in_node,
                    out_node=out_node,
                    innov=innov_num,
                    weight=weight,
                    enabled=True
                )
        
        ind.compile_network()
        pop.individuals.append(ind)
        
    pop.reset_generation_innovations()
    
    return pop

In [8]:
def cluster_species(individuals: list, previous_representatives: dict, compatibility_threshold: float = 3.0) -> list:
    """
    Greedy sequential clustering.
    """
    representatives = previous_representatives.copy() 
    clusters = []

    next_cluster_id = max(representatives.keys()) + 1 if representatives else 0
    
    for ind in individuals:
        assigned_cluster = None

        for cluster_id, rep in representatives.items():
            if ind.distance(rep) < compatibility_threshold:
                assigned_cluster = cluster_id
                break 
                
        if assigned_cluster is None:
            assigned_cluster = next_cluster_id
            representatives[next_cluster_id] = ind 
            next_cluster_id += 1
            
        clusters.append(assigned_cluster)
        
    return clusters

In [78]:
def train_NEAT(gym_env_func: Callable, action_mapper: Callable, max_steps:int, n_runs:int, max_iterations:int, population_size:int, compatibility_threshold:float):
    best_individual = None
    sample_env = gym_env_func()
    n_inputs = sample_env.observation_space.shape[0]

    if isinstance(sample_env.action_space, gym.spaces.Discrete):
        n_outputs = sample_env.action_space.n
    else:
        n_outputs = sample_env.action_space.shape[0]

    envs = gym.vector.AsyncVectorEnv(
        [lambda: gym_env_func() for _ in range(population_size)]
    )
    population = create_population(
        size=population_size, 
        n_inputs=n_inputs, 
        n_outputs=n_outputs,
        compatibility_threshold=compatibility_threshold,
        cluster_func=cluster_species # hidden dependency
    )

    for i in range(max_iterations):
        population.evaluate_fitness(
            vector_env=envs, 
            action_mapper=action_mapper, 
            max_steps=max_steps,
            n_runs=n_runs,
        )
        population.individuals.sort(key=lambda ind: ind.fitness, reverse=True)
        if i % 1 == 0:
            print(f"Generation {i}: Best fitness = {population.individuals[0].fitness}")
        if best_individual is None or population.individuals[0].fitness > best_individual.fitness:
            best_individual = population.individuals[0].clone()

        population.select_reproduce_mutate()

    envs.close()
    return best_individual

# Running NEAT in gymnasium
Note: when running CartPole or LunarLander use argmax and sigmoid activation function, for BipedalWalker use tanh (maybe sin) and contious action mapper.

In [14]:
import time

def watch_agent(model, render_env, action_mapper):
    state, _ = render_env.reset()
    done = False
    steps = 0
    
    while not done:
        output = model.forward_pass(state)
        action = action_mapper(output)
    
        next_state, reward, terminated, truncated, _ = render_env.step(action)
        done = terminated or truncated
        state = next_state
        steps += 1
        time.sleep(0.005)

    print("Survived", steps, "steps")
    render_env.close()

In [11]:
def argmax_action_mapper(network_output):
    return np.argmax(network_output)

## CartPole

In [77]:
import gymnasium as gym

make_cartpole_gym_env = lambda : gym.make("CartPole-v1")

best_cartpole_model = train_NEAT(
    gym_env_func=make_cartpole_gym_env,
    action_mapper=argmax_action_mapper,
    max_steps=500,
    n_runs=2,
    max_iterations=2,
    population_size=100,
    compatibility_threshold=3.0
)

Generation 0: Best fitness = 500.0
Generation 1: Best fitness = 500.0


In [91]:
watch_agent(
    model=best_cartpole_model,
    render_env=gym.make('CartPole-v1', render_mode="human"),
    action_mapper=argmax_action_mapper,
)

Survived 500 steps


## Lunar Lander
Note: the agent often doesn't stop, maybe increase pentalty for taking to long?

In [79]:
make_lunar_env = lambda: gym.make('LunarLander-v3')

best_lunar_model = train_NEAT(
    gym_env_func=make_lunar_env,
    action_mapper=argmax_action_mapper,
    max_steps=1000,
    n_runs=3,
    max_iterations=20,
    population_size=150,
    compatibility_threshold=3.0,
)

Generation 0: Best fitness = -96.67504287558666
Generation 1: Best fitness = -56.75851229130262
Generation 2: Best fitness = -29.354550983090007
Generation 3: Best fitness = -11.930888255119958
Generation 4: Best fitness = -29.43584425713378
Generation 5: Best fitness = -37.76798248303623
Generation 6: Best fitness = 33.9610415697949
Generation 7: Best fitness = 46.23459991105051
Generation 8: Best fitness = 52.02833160296323
Generation 9: Best fitness = 22.079230134776452
Generation 10: Best fitness = 139.78162681958406
Generation 11: Best fitness = 94.63214028880903
Generation 12: Best fitness = 86.64745326299612
Generation 13: Best fitness = 159.16942477256717
Generation 14: Best fitness = 83.041407726619
Generation 15: Best fitness = 199.159417060508
Generation 16: Best fitness = 181.63785501880633
Generation 17: Best fitness = 220.98465428826353
Generation 18: Best fitness = 254.8254091529011
Generation 19: Best fitness = 184.06503658008702


In [95]:
watch_agent(
    model=best_lunar_model,
    render_env=gym.make('LunarLander-v3', render_mode="human"),
    action_mapper=argmax_action_mapper,
)

Survived 292 steps


## Bipedal Walker

In [35]:
# note to switch activation function from sigmoid [0, 1] to tanh [-1, 1]
def continuous_action_mapper(network_output):
    return np.array(network_output)

In [99]:
make_bipedal_env = lambda: gym.make('BipedalWalker-v3')

best_bipedal_model = train_NEAT(
    gym_env_func=make_bipedal_env,
    action_mapper=continuous_action_mapper, 
    max_steps=800,
    n_runs=1,
    max_iterations=1,
    population_size=150,
    compatibility_threshold=3.0,
)

Generation 0: Best fitness = -36.51122676020323


In [100]:
watch_agent(
    model=best_bipedal_model,
    render_env=gym.make('BipedalWalker-v3', render_mode="human"),
    action_mapper=continuous_action_mapper,
)

Survived 1600 steps
